In [1]:
import pandas as pd
import geopandas as gpd
import tobler   
import numpy as np

In [3]:
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.15f' % x)

In [4]:
sectors_2022 = gpd.read_file('../../dataset/census_sectors/setores_censitarios_pop_fcu.shp')
sectors_2022.set_crs("EPSG:4674", inplace=True)
sectors_2022.columns = ['CD_GEOCOD_2022', 'pop_2022', 'CD_FCU_2022', 'geometry']
sectors_2022

,CD_GEOCOD_2022,pop_2022,CD_FCU_2022,geometry
0,120001305000001,920,None,"POLYGON ((-67.04925 -10.07069, -67.04938 -10.0..."
1,120001305000002,354,None,"POLYGON ((-67.03937 -10.07846, -67.03897 -10.0..."
2,120001305000005,278,None,"POLYGON ((-67.03598 -10.00512, -67.03653 -10.0..."
3,120001305000006,401,None,"POLYGON ((-66.91526 -9.86712, -66.91544 -9.867..."
4,120001305000007,241,None,"POLYGON ((-66.88699 -9.83206, -66.88808 -9.832..."
...,...,...,...,...
489073,172210705000030,276,17221070001,"POLYGON ((-48.53139 -6.41658, -48.53136 -6.416..."
489074,172210705000031,289,None,"POLYGON ((-48.52973 -6.41638, -48.52999 -6.416..."
489075,172210705000032,152,None,"POLYGON ((-48.53265 -6.41567, -48.53287 -6.415..."
489076,172210705000034,393,None,"POLYGON ((-48.53492 -6.41734, -48.53513 -6.417..."


In [5]:
sectors_2010 = gpd.read_file('../../dataset/census_sectors_2010/setores_censitarios_pop_ibp.shp')
sectors_2010.set_crs("EPSG:4674", inplace=True)
sectors_2010.columns = ['ID', 'CD_GEOCOD_2010', 'pop_2010','IBP_2010', 'geometry']
sectors_2010

,ID,CD_GEOCOD_2010,pop_2010,IBP_2010,geometry
0,17182,110009812000003,555.000000000000000,3.128591759064130,"POLYGON ((-60.89575 -11.35508, -60.89557 -11.3..."
1,17183,110009815000001,53.000000000000000,2.838013524675720,"POLYGON ((-60.74999 -11.3999, -60.74913 -11.40..."
2,17184,110009815000002,507.000000000000000,2.903041567777580,"POLYGON ((-60.72986 -11.35738, -60.72954 -11.3..."
3,17185,110009815000003,555.000000000000000,3.543457095292630,"POLYGON ((-60.91829 -11.29374, -60.916 -11.292..."
4,17186,110009815000004,336.000000000000000,3.319935981328950,"POLYGON ((-60.69047 -11.38391, -60.6903 -11.38..."
...,...,...,...,...,...
316569,4450,530010805300153,434.000000000000000,-2.459134180570260,"POLYGON ((-47.81165 -15.86005, -47.80981 -15.8..."
316570,4451,530010805300154,534.000000000000000,-3.100081638167780,"POLYGON ((-47.81951 -15.8618, -47.81866 -15.85..."
316571,4452,530010805300155,532.000000000000000,-2.973201604504560,"POLYGON ((-47.81758 -15.85556, -47.81269 -15.8..."
316572,4453,530010805300156,2103.000000000000000,-0.975027283141378,"POLYGON ((-47.7841 -15.90137, -47.78145 -15.90..."


# Dasymetric Interpolation - Pop 2022 ancillary

In [ ]:
# Create an intersection between the 2010 and 2022 sectors
intersection = gpd.overlay(sectors_2010, sectors_2022, how="intersection")
intersection

/var/folders/fx/kx7__pwn2tv9449n14s5rx1w0000gn/T/ipykernel_14533/2535126458.py:2: UserWarning: `keep_geom_type=True` in overlay resulted in 30034 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  intersection = gpd.overlay(sectors_2010, sectors_2022, how="intersection")


,ID,CD_GEOCOD_2010,pop_2010,IBP_2010,CD_GEOCOD_2022,pop_2022,CD_FCU_2022,geometry
0,17182,110009812000003,555.000000000000000,3.128591759064130,110009805000022,616,None,"MULTIPOLYGON (((-60.83109 -11.48749, -60.83352..."
1,17182,110009812000003,555.000000000000000,3.128591759064130,110009805000083,495,None,"POLYGON ((-60.99222 -11.5001, -60.9796 -11.483..."
2,17182,110009812000003,555.000000000000000,3.128591759064130,110009812000001,173,None,"MULTIPOLYGON (((-60.92784 -11.4616, -60.92784 ..."
3,17182,110009812000003,555.000000000000000,3.128591759064130,110009812000002,714,None,"MULTIPOLYGON (((-60.98469 -11.47198, -60.96361..."
4,17182,110009812000003,555.000000000000000,3.128591759064130,110009812000003,391,None,"POLYGON ((-60.89413 -11.35596, -60.89401 -11.3..."
...,...,...,...,...,...,...,...,...
2179385,4454,530010805300158,169.000000000000000,NaN,530010805300273,221,None,"POLYGON ((-47.7822 -15.91286, -47.7822 -15.912..."
2179386,4454,530010805300158,169.000000000000000,NaN,530010805300274,206,None,"POLYGON ((-47.7861 -15.92517, -47.7862 -15.925..."
2179387,4454,530010805300158,169.000000000000000,NaN,530010805400043,3586,None,"MULTIPOLYGON (((-47.78323 -15.92072, -47.78462..."
2179388,4454,530010805300158,169.000000000000000,NaN,530010805400082,0,None,"POLYGON ((-47.78991 -15.91907, -47.78883 -15.9..."


In [ ]:
# Calculate the area of the intersection
intersection["area_intersection"] = intersection.geometry.area
sectors_2022["area_total"] = sectors_2022.geometry.area

# Merge the IBP and population data from 2010
intersection = intersection.merge(sectors_2022[["CD_GEOCOD_2022", "area_total"]], on="CD_GEOCOD_2022")
intersection

/var/folders/fx/kx7__pwn2tv9449n14s5rx1w0000gn/T/ipykernel_14533/1367435111.py:2: UserWarning: Geometry is in a geographic CRS. Results from 'area' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  intersection["area_intersection"] = intersection.geometry.area
/var/folders/fx/kx7__pwn2tv9449n14s5rx1w0000gn/T/ipykernel_14533/1367435111.py:3: UserWarning: Geometry is in a geographic CRS. Results from 'area' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  sectors_2022["area_total"] = sectors_2022.geometry.area


,ID,CD_GEOCOD_2010,pop_2010,IBP_2010,CD_GEOCOD_2022,pop_2022,CD_FCU_2022,geometry,area_intersection,area_total_x,pop_prop,weight,IBP_prop,area_total_y,area_total
0,17182,110009812000003,555.000000000000000,3.128591759064130,110009805000022,616,None,"MULTIPOLYGON (((-60.83109 -11.48749, -60.83352...",0.000134314983028,0.016272913676220,5.084401674566425,0.008253898822348,0.025823079835747,0.016272913676220,0.016272913676220
1,17182,110009812000003,555.000000000000000,3.128591759064130,110009805000083,495,None,"POLYGON ((-60.99222 -11.5001, -60.9796 -11.483...",0.000023047712370,0.003834371553035,2.975355274089692,0.006010818735535,0.018805397961222,0.003834371553035,0.003834371553035
2,17182,110009812000003,555.000000000000000,3.128591759064130,110009812000001,173,None,"MULTIPOLYGON (((-60.92784 -11.4616, -60.92784 ...",0.000000000282452,0.000033428758050,0.001461742281544,0.000008449377350,0.000026434652346,0.000033428758050,0.000033428758050
3,17182,110009812000003,555.000000000000000,3.128591759064130,110009812000002,714,None,"MULTIPOLYGON (((-60.98469 -11.47198, -60.96361...",0.000169442255247,0.018970602220249,6.377328924071543,0.008931833226991,0.027944059827299,0.018970602220249,0.018970602220249
4,17182,110009812000003,555.000000000000000,3.128591759064130,110009812000003,391,None,"POLYGON ((-60.89413 -11.35596, -60.89401 -11.3...",0.009471774248500,0.009792548871960,378.192009004755732,0.967242989781984,3.026108446844466,0.009792548871960,0.009792548871960
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3478921,4454,530010805300158,169.000000000000000,NaN,530010805300273,221,None,"POLYGON ((-47.7822 -15.91286, -47.7822 -15.912...",0.000000000000010,0.000040495238615,0.000000054523550,0.000000000246713,NaN,0.000040495238615,0.000040495238615
3478922,4454,530010805300158,169.000000000000000,NaN,530010805300274,206,None,"POLYGON ((-47.7861 -15.92517, -47.7862 -15.925...",0.000001464299972,0.000483515036415,0.623860214225948,0.003028447641874,NaN,0.000483515036415,0.000483515036415
3478923,4454,530010805300158,169.000000000000000,NaN,530010805400043,3586,None,"MULTIPOLYGON (((-47.78323 -15.92072, -47.78462...",0.000000000101432,0.000008277839980,0.043940639087766,0.000012253385133,NaN,0.000008277839980,0.000008277839980
3478924,4454,530010805300158,169.000000000000000,NaN,530010805400082,0,None,"POLYGON ((-47.78991 -15.91907, -47.78883 -15.9...",0.000144912377792,0.000203393707045,0.000000000000000,NaN,NaN,0.000203393707045,0.000203393707045


In [ ]:
# Calculate the proportion of the population in the overlapping area
intersection["pop_prop"] = intersection["pop_2022"] * (intersection["area_intersection"] / intersection["area_total"])
intersection['weight'] = intersection["pop_prop"] / intersection["pop_2022"]

# Adjust IBP considering population
intersection["IBP_prop"] = intersection["IBP_2010"] * intersection["weight"]
intersection

,ID,CD_GEOCOD_2010,pop_2010,IBP_2010,CD_GEOCOD_2022,pop_2022,CD_FCU_2022,geometry,area_intersection,area_total_x,pop_prop,weight,IBP_prop,area_total_y,area_total
0,17182,110009812000003,555.000000000000000,3.128591759064130,110009805000022,616,None,"MULTIPOLYGON (((-60.83109 -11.48749, -60.83352...",0.000134314983028,0.016272913676220,5.084401674566425,0.008253898822348,0.025823079835747,0.016272913676220,0.016272913676220
1,17182,110009812000003,555.000000000000000,3.128591759064130,110009805000083,495,None,"POLYGON ((-60.99222 -11.5001, -60.9796 -11.483...",0.000023047712370,0.003834371553035,2.975355274089692,0.006010818735535,0.018805397961222,0.003834371553035,0.003834371553035
2,17182,110009812000003,555.000000000000000,3.128591759064130,110009812000001,173,None,"MULTIPOLYGON (((-60.92784 -11.4616, -60.92784 ...",0.000000000282452,0.000033428758050,0.001461742281544,0.000008449377350,0.000026434652346,0.000033428758050,0.000033428758050
3,17182,110009812000003,555.000000000000000,3.128591759064130,110009812000002,714,None,"MULTIPOLYGON (((-60.98469 -11.47198, -60.96361...",0.000169442255247,0.018970602220249,6.377328924071543,0.008931833226991,0.027944059827299,0.018970602220249,0.018970602220249
4,17182,110009812000003,555.000000000000000,3.128591759064130,110009812000003,391,None,"POLYGON ((-60.89413 -11.35596, -60.89401 -11.3...",0.009471774248500,0.009792548871960,378.192009004755732,0.967242989781984,3.026108446844466,0.009792548871960,0.009792548871960
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3478921,4454,530010805300158,169.000000000000000,NaN,530010805300273,221,None,"POLYGON ((-47.7822 -15.91286, -47.7822 -15.912...",0.000000000000010,0.000040495238615,0.000000054523550,0.000000000246713,NaN,0.000040495238615,0.000040495238615
3478922,4454,530010805300158,169.000000000000000,NaN,530010805300274,206,None,"POLYGON ((-47.7861 -15.92517, -47.7862 -15.925...",0.000001464299972,0.000483515036415,0.623860214225948,0.003028447641874,NaN,0.000483515036415,0.000483515036415
3478923,4454,530010805300158,169.000000000000000,NaN,530010805400043,3586,None,"MULTIPOLYGON (((-47.78323 -15.92072, -47.78462...",0.000000000101432,0.000008277839980,0.043940639087766,0.000012253385133,NaN,0.000008277839980,0.000008277839980
3478924,4454,530010805300158,169.000000000000000,NaN,530010805400082,0,None,"POLYGON ((-47.78991 -15.91907, -47.78883 -15.9...",0.000144912377792,0.000203393707045,0.000000000000000,NaN,NaN,0.000203393707045,0.000203393707045


In [ ]:
# Sum the adjusted IBP for each 2022 sector
ibp_2022 = intersection.groupby("CD_GEOCOD_2022")["IBP_prop"].sum().reset_index()

# Merge with the 2022 shapefile
sectors_2022 = sectors_2022.merge(ibp_2022, on="CD_GEOCOD_2022", how="left")

# Save the new shapefile with the interpolated IBP
sectors_2022.to_file('../../dataset/census_sectors/setores_censitarios_pop_fcu_ibp.shp')

/var/folders/fx/kx7__pwn2tv9449n14s5rx1w0000gn/T/ipykernel_14533/3979698974.py:8: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  sectors_2022.to_file('../../dataset/census_sectors/setores_censitarios_pop_fcu_ibp.shp')
/Users/joaopedro/workspace/doutorado/state_of_brazilian_favelas/venv/lib/python3.13/site-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'CD_GEOCOD_2022' to 'CD_GEOCOD_'
  ogr_write(
/Users/joaopedro/workspace/doutorado/state_of_brazilian_favelas/venv/lib/python3.13/site-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'CD_FCU_2022' to 'CD_FCU_202'
  ogr_write(
